[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Errors and Retries


## What you will be able to do

Set a timeout on a request and say what each part of it limits, tell apart the ways a request fails
without a response, decide which failures are worth trying again, retry with backoff inside a
deadline, and hand the retrying to urllib3's `Retry` when that is enough.


## The idea

### The problem

Every failure so far came with a response: a `404` saying a station was not there, a `401` asking for
a key, a `429` asking for patience. Some failures come with none. A server takes too long to answer,
a connection closes before a response arrives, or the network between two computers drops for a
moment and recovers. With no status code to return, requests raises an exception, and a program that
does not expect one stops.

Many of these failures pass on their own, and sending the request again is the fix. While this guide
was being built, requests from GitHub's servers to Open-Meteo twice failed at the connection, and the
same requests went through when they were sent again. Sending a request again is not always right,
though. Retrying a `404` only repeats it, retrying too soon adds to the load that made a server fail,
and retrying a request that changes something can change it twice.

### What errors and retries are

> A request fails in one of two ways. With a **response**, the status code says what went wrong, as
> the **Status Codes** notebook showed. Without one, requests raises an exception: a **timeout**, when
> the server takes longer than the client agreed to wait, or a **connection error**, when a connection
> cannot be made or closes before a response arrives. A **transient** failure can pass on its own, and
> a **retry** sends the same request again after one. A request is safe to retry when sending it twice
> has the same effect as sending it once, which HTTP calls **idempotent**.

### Why it works that way

- **A timeout is a limit a client sets for itself.** Without one, requests waits as long as a server
  takes, which can be forever. `timeout` limits the wait for a connection, and then the wait between
  one piece of the response and the next, but never the whole request.
- **No response is not the same as nothing done.** A refused connection never reached a server. A
  timeout, or a connection that closed mid-request, can come after the server did the work, and the
  client cannot tell whether it did.
- **Some failures pass, and some do not.** A timeout, a closed connection and a `502`, `503` or `504`
  often pass. A `400`, `401`, `403` or `404` comes back the same however often the request is sent.
- **A request that only reads is safe to send twice.** Asking twice for a station changes nothing. A
  request that places an order can place two when its response is lost, which the **Sending Data**
  notebook deals with.
- **Retries wait, and stop.** Backoff with jitter, from the **Rate Limits** notebook, spaces the
  attempts. A most on the attempts ends a loop that would never succeed, and a deadline caps the time
  all of them together may take.

### Where you will meet this

While this guide was being built, two runs on GitHub's servers failed on Open-Meteo at the connection,
once during the TLS handshake and once waiting for a response, while the same requests took under
half a second from elsewhere. A longer timeout would not have helped and a fresh attempt did, so the
guide's build now runs a notebook again when its failure names a network error. requests'
documentation says that it does not retry failed connections by default, and shows how to add
retries with urllib3's `Retry`, which retries only the methods HTTP calls idempotent unless told
otherwise. It also advises a connect timeout slightly larger than a multiple of 3 seconds, the time
TCP waits before it sends a lost packet again.

### What this notebook covers

- A request that takes too long, `ReadTimeout`, and the two parts of a timeout
- A connection that is refused or closed, and `ConnectionError`
- The exceptions requests raises, and the one class that catches all of them
- Which failures are worth retrying, and which are not
- A retry loop that sends the same request again, with backoff, inside a deadline
- urllib3's `Retry`, which retries inside a `Session`
- A client that times out, retries what is worth retrying, and stops in time
- Five errors, from a `ConnectionError` that went uncaught to a timeout taken for a deadline

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

for url in ["http://127.0.0.1:8765/network/report", "http://127.0.0.1:8765/hang-up"]:
    try:
        response = requests.get(url, timeout=1)
        print(response.status_code)
    except requests.exceptions.RequestException as error:
        print(type(error).__name__, "|", error)
```

```
ReadTimeout | HTTPConnectionPool(host='127.0.0.1', port=8765): Read timed out. (read timeout=1)
ConnectionError | ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
```

Two requests, and no response to either: one waited longer than it was allowed, and the other's
connection was closed. requests raised an exception for each, and one `except` caught both.


## Setup

Eleven imports, the last of them the practice API.

- `requests` sends every request, and `requests.exceptions` holds what it raises
- `HTTPAdapter` sends a `Session`'s requests, and can retry them
- `Retry`, from urllib3, the library requests is built on, says what to retry and how long to wait
- `time` measures with `monotonic`, and waits with `sleep`
- `random` chooses waits at random, from a seeded generator
- `uuid` makes an id for a request, kept the same on every attempt
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, started in the background by `start()`

Several examples wait on purpose, for a slow response or between attempts, so they take a few seconds.


In [1]:
import importlib
import random
import sys
import time
import urllib.request
import uuid
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### A request that takes too long: timeouts

`/network/report` builds a small report of the network, and takes 2 seconds to do it. A client that
waits at most 1 second gets an exception instead:


In [2]:
start = time.monotonic()
try:
    requests.get(f"{BASE}/network/report", timeout=1)
except requests.exceptions.Timeout as error:
    print(type(error).__name__, "after", round(time.monotonic() - start), "second |", error)

response = requests.get(f"{BASE}/network/report", timeout=5)
print(response.status_code, response.json())


ReadTimeout after 1 second | HTTPConnectionPool(host='127.0.0.1', port=8765): Read timed out. (read timeout=1)
200 {'stations': 4, 'reporting': 3, 'readings': 216, 'events': 54}


`ReadTimeout` means the connection was made and the request sent, and the response did not start
within the second allowed. With 5 seconds, the same request succeeded. A timeout has two parts, which
a pair sets separately, as in `timeout=(3.05, 10)`. The first limits the wait for a connection, and
raises `ConnectTimeout` when a server cannot be reached in that time. The second limits the wait for
the response, and then the wait between one piece of it and the next. A single number sets both.
When a timeout runs out, a program either gives up or tries again, and the rest of this notebook is
about choosing.

### A connection that fails: ConnectionError

A request can also fail before any response starts. Port 9 of this computer has nothing listening
on it, as port 8799 had in the **What an API Is** notebook, and `/hang-up` receives a request and then
closes the connection without answering:


In [3]:
for label, url in [("nothing listening", "http://127.0.0.1:9/"), ("hung up", f"{BASE}/hang-up")]:
    try:
        requests.get(url, timeout=5)
    except requests.exceptions.ConnectionError as error:
        print(f"{label:<17} {type(error).__name__}")

try:
    requests.get(f"{BASE}/hang-up", timeout=5)
except requests.exceptions.ConnectionError as error:
    print(error)


nothing listening ConnectionError
hung up           ConnectionError
('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


Both are `ConnectionError`, and they are different failures. Nothing was listening on port 9, so the
request never reached a server, and sending it again can do no harm. `/hang-up` received the request
before it closed the connection, and a server that behaves like that may already have done what the
request asked. The refused connection's message names the operating system's code for the failure,
which differs between computers, so the cell printed only its class. A host name that does not exist
fails the same way, sometimes only after the computer has waited a long time for an answer about the
name.

### The exceptions requests raises

requests arranges its exceptions in a family, so one `except` can catch a single kind of failure or
every kind. `issubclass` shows which class catches which:


In [4]:
families = [requests.exceptions.Timeout, requests.exceptions.ConnectionError, requests.exceptions.HTTPError,
            requests.exceptions.RequestException]
for error in [requests.exceptions.ReadTimeout, requests.exceptions.ConnectTimeout, requests.exceptions.ConnectionError,
              requests.exceptions.HTTPError, requests.exceptions.JSONDecodeError, requests.exceptions.RetryError]:
    caught_by = [family.__name__ for family in families if issubclass(error, family)]
    print(f"{error.__name__:<16} caught by {', '.join(caught_by)}")


ReadTimeout      caught by Timeout, RequestException
ConnectTimeout   caught by Timeout, ConnectionError, RequestException
ConnectionError  caught by ConnectionError, RequestException
HTTPError        caught by HTTPError, RequestException
JSONDecodeError  caught by RequestException
RetryError       caught by RequestException


`RequestException` catches everything requests raises, and the narrower classes let a program treat
one failure differently from another. `ConnectTimeout` is both a `Timeout` and a `ConnectionError`,
since the connection was never made. `HTTPError` is what `raise_for_status` raises for a `4xx` or a
`5xx`, `JSONDecodeError` comes from `response.json()` on a body that is not JSON, and `RetryError`
comes from urllib3's `Retry`, later in this notebook.

### Which failures are worth retrying

The **Status Codes** notebook's `decide` said what to do about a response. A retry loop needs one
question answered about every outcome, whether a response or an exception: could the same `GET`,
sent again, succeed? `outcome_of` sends a request and returns whichever came back, `describe` names
it, and `worth_retrying` answers the question:


In [5]:
RETRY_STATUSES = {408, 429, 500, 502, 503, 504}


def outcome_of(url, **kwargs):
    """The response to a GET, or the exception raised in its place."""
    try:
        return requests.get(url, **kwargs)
    except requests.exceptions.RequestException as error:
        return error


def describe(outcome):
    return outcome.status_code if isinstance(outcome, requests.Response) else type(outcome).__name__


def worth_retrying(outcome):
    """Whether a GET that ended in this outcome could succeed if it were sent again."""
    if isinstance(outcome, requests.Response):
        return outcome.status_code in RETRY_STATUSES
    return isinstance(outcome, (requests.exceptions.Timeout, requests.exceptions.ConnectionError))


cases = [("a slow report", f"{BASE}/network/report", 1), ("a closed connection", f"{BASE}/hang-up", 5),
         ("a gateway failure", f"{BASE}/status/502", 5), ("maintenance", f"{BASE}/status/503", 5),
         ("a server bug", f"{BASE}/status/500", 5), ("a missing station", f"{BASE}/stations/narvik", 5),
         ("no API key", f"{BASE}/me", 5)]
for label, url, timeout in cases:
    outcome = outcome_of(url, timeout=timeout)
    print(f"{label:<20} {describe(outcome)!s:<16} worth retrying: {worth_retrying(outcome)}")


a slow report        ReadTimeout      worth retrying: True
a closed connection  ConnectionError  worth retrying: True
a gateway failure    502              worth retrying: True
maintenance          503              worth retrying: True
a server bug         500              worth retrying: True
a missing station    404              worth retrying: False
no API key           401              worth retrying: False


A timeout, a closed connection and the `5xx` codes are worth another try, and so are
`408 Request Timeout` and `429 Too Many Requests`, which both ask for one. `500` is the least likely
to pass, because a bug that fails on every request also answers `500`, so a few attempts are enough.
A `404` and a `401` stay the same however many times they are sent. `JSONDecodeError` is not on the
list: a body that is not JSON is usually a gateway's error page, and the status code that came with it
has already decided. All of this is for a `GET`, which only reads. For a request that changes
something, the answer also depends on whether the server may already have done it.

### A retry loop, with backoff and a deadline

A retry loop sends the same request again while the outcome is worth retrying. It waits between
attempts with backoff and jitter, as the **Rate Limits** notebook did, and stops at a most number of
attempts, or when a wait would carry it past a deadline for all the attempts together. It also sends
the same `X-Request-Id` on every attempt, which lets a server tell that the attempts are one request.
`/network/unstable` counts attempts that way: it closes the first attempt's connection, answers the
second with `503`, and answers the third. `uuid.uuid4().hex` makes a random id that no other request
will have:


In [6]:
def backoff(attempt, rng, cap=30):
    """Seconds to wait after failed attempt number attempt + 1: a random share of a longest that doubles."""
    return rng.uniform(0, min(cap, 2 ** attempt))


def get_with_retries(url, attempts=4, deadline=10, timeout=5, rng=None):
    """The response to a GET, sent again after each failure worth retrying, within the attempts and the deadline."""
    rng = rng or random.Random()
    headers = {"X-Request-Id": uuid.uuid4().hex}           # the same id on every attempt
    stop_at = time.monotonic() + deadline
    for attempt in range(1, attempts + 1):
        outcome = outcome_of(url, headers=headers, timeout=timeout)
        print(f"  attempt {attempt}: {describe(outcome)}")
        if not worth_retrying(outcome) or attempt == attempts:
            break
        wait = backoff(attempt - 1, rng)
        if time.monotonic() + wait > stop_at:
            print("  stopping: the next wait would pass the deadline")
            break
        time.sleep(wait)
    if isinstance(outcome, Exception):
        raise outcome
    return outcome


response = get_with_retries(f"{BASE}/network/unstable", rng=random.Random(12))
print(response.status_code, "on attempt", response.json()["attempt"])


  attempt 1: ConnectionError
  attempt 2: 503
  attempt 3: 200
200 on attempt 3


The first attempt's connection closed, the second got `503`, and the third succeeded, with a random
wait before each retry. A client that made a new id for every attempt would meet the closed
connection every time, because the server would take every attempt for a first one. A deadline
matters when every attempt is slow. Here the report takes 2 seconds, the read timeout is 1, and the
attempts together have 3 seconds:


In [7]:
try:
    get_with_retries(f"{BASE}/network/report", attempts=5, deadline=3, timeout=1, rng=random.Random(28))
except requests.exceptions.Timeout as error:
    print("gave up:", type(error).__name__)


  attempt 1: ReadTimeout
  attempt 2: ReadTimeout
  attempt 3: ReadTimeout
  stopping: the next wait would pass the deadline
gave up: ReadTimeout


Three attempts of a second each, and the waits between them, came close enough to 3 seconds that the
next wait would have passed the deadline, so the loop stopped and raised the last timeout. Without a
deadline, five attempts and their waits could have taken up to twenty seconds, for a report that
would never have arrived in time.

### Retries a library makes: urllib3's Retry

A `requests.Session` sends every request through an adapter, and an `HTTPAdapter` given a `Retry`
from urllib3, the library requests is built on, retries inside the request, before the program sees
a response. `mount` attaches the adapter to every address that starts with `http://`:


In [8]:
retry = Retry(total=3, backoff_factor=0.5, status_forcelist=[502, 503, 504])
session = requests.Session()
session.mount("http://", HTTPAdapter(max_retries=retry))

response = session.get(f"{BASE}/network/unstable", headers={"X-Request-Id": uuid.uuid4().hex}, timeout=5)
print(response.status_code, "on attempt", response.json()["attempt"])
for failure in response.raw.retries.history:
    print("  before it:", failure.error or failure.status)


200 on attempt 3
  before it: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  before it: 503


`total=3` allows three retries after the first attempt. Connection errors and timeouts are retried
without being named, and `status_forcelist` adds the status codes worth retrying. A `503` or `429`
with `Retry-After` gets the wait it asks for, which here was 1 second. For other failures,
`backoff_factor` sets waits that double: none before the first retry, then 1 second, then 2, and so
on, and `backoff_jitter` can add a random part. By default `Retry` retries only the methods HTTP calls
idempotent: `GET`, `HEAD`, `OPTIONS`, `PUT`, `DELETE` and `TRACE`. `response.raw` is urllib3's own
response, whose `retries.history` records every failure before the attempt that succeeded.

### A client that retries what is worth retrying

The pieces of this notebook, in one client. `RetryingClient` sets both parts of its timeout, keeps
one request id across the attempts of a request, retries only what is worth retrying, waits as
`Retry-After` asks or with backoff and jitter, and stops at a most number of attempts, or before a
wait that would pass its deadline. Instead of printing, it keeps a log of every attempt:


In [9]:
class RetryingClient:
    """GETs from one API that time out, retry the failures worth retrying, and stop by a deadline."""

    def __init__(self, base, timeout=(3.05, 5), attempts=4, deadline=15, seed=None):
        self.base = base
        self.timeout = timeout
        self.attempts = attempts
        self.deadline = deadline
        self.rng = random.Random(seed)
        self.log = []

    def wait_before_retry(self, outcome, attempt):
        retry_after = outcome.headers.get("Retry-After", "") if isinstance(outcome, requests.Response) else ""
        return int(retry_after) if retry_after.isdigit() else backoff(attempt - 1, self.rng)

    def get(self, path, **params):
        """The JSON at path, after as many attempts as its failures, the attempts and the deadline allow."""
        headers = {"X-Request-Id": uuid.uuid4().hex}
        stop_at = time.monotonic() + self.deadline
        for attempt in range(1, self.attempts + 1):
            outcome = outcome_of(f"{self.base}{path}", params=params, headers=headers, timeout=self.timeout)
            self.log.append(f"{path}, attempt {attempt}: {describe(outcome)}")
            if not worth_retrying(outcome) or attempt == self.attempts:
                break
            wait = self.wait_before_retry(outcome, attempt)
            if time.monotonic() + wait > stop_at:
                self.log.append(f"{path}: stopped, since the next wait would pass the deadline")
                break
            time.sleep(wait)
        if isinstance(outcome, Exception):
            raise outcome
        outcome.raise_for_status()
        return outcome.json()


client = RetryingClient(BASE, timeout=(3.05, 1), deadline=4, seed=30)
print("readings from attempt", client.get("/network/unstable")["attempt"])
for path in ["/stations/narvik", "/network/report"]:
    try:
        client.get(path)
    except requests.exceptions.RequestException as error:
        print(f"{path}: gave up with {type(error).__name__}")

for line in client.log:
    print(" ", line)


readings from attempt 3
/stations/narvik: gave up with HTTPError
/network/report: gave up with ReadTimeout
  /network/unstable, attempt 1: ConnectionError
  /network/unstable, attempt 2: 503
  /network/unstable, attempt 3: 200
  /stations/narvik, attempt 1: 404
  /network/report, attempt 1: ReadTimeout
  /network/report, attempt 2: ReadTimeout
  /network/report, attempt 3: ReadTimeout
  /network/report: stopped, since the next wait would pass the deadline


### Where each part came from

| In the client | What it relies on | The section that showed it |
|---|---|---|
| `timeout=(3.05, 5)` | a wait for the connection and a wait for the response, set apart | A request that takes too long: timeouts |
| `outcome_of(...)` and `describe(outcome)` | an exception kept as an outcome, like a response | Which failures are worth retrying |
| `worth_retrying(outcome)` | timeouts, connection errors and some codes retried, and a `404` not | Which failures are worth retrying |
| one `X-Request-Id` for every attempt | attempts a server can tell are one request | A retry loop, with backoff and a deadline |
| `backoff(attempt - 1, self.rng)` and `stop_at` | random waits under a longest that doubles, inside a deadline | A retry loop, with backoff and a deadline |
| `int(retry_after)` | the wait a `503` or a `429` asks for | the **Rate Limits** notebook |
| `raise outcome` and `raise_for_status()` | a failure the program still sees, once the retries are over | The exceptions requests raises |

The unstable endpoint took three attempts: a wait from backoff after the closed connection, and the
1 second that `Retry-After` asked for after the `503`. The missing station got one attempt, since a `404`
cannot pass, and the report got three, each ending in a timeout, before the next wait would have
passed the 4-second deadline. Every request the client gave up on raised what its last attempt met,
so the program can still tell a missing station from a slow server.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/12-errors-and-retries-solutions.ipynb).

**1.** Request `/network/report` with a timeout of 3 seconds, and print the status code and how many
whole seconds the request took.


In [10]:
# your code here


**2.** Request `/hang-up`, catch the exception with the narrowest `except` that catches it, and print
its class name and whether it is also a `requests.exceptions.RequestException`.


In [11]:
# your code here


**3.** Use `outcome_of`, `describe` and `worth_retrying` to print, for `/status/504`, `/status/408`,
`/status/400`, and `/trickle` with a timeout of 1, what came back and whether it is worth retrying.


In [12]:
# your code here


**4.** Send `/network/unstable` three requests by hand, all with one `X-Request-Id`, and print what each
got. Then send one more with a new id, and print what it got.


In [13]:
# your code here


**5.** Mount an `HTTPAdapter` with `Retry(total=1, status_forcelist=[503])` on a new `Session`, request
`/network/unstable` through it with a new `X-Request-Id`, and print what happens.


In [14]:
# your code here


**6.** Use `get_with_retries` with `attempts=3` and `random.Random(5)` to request `/status/500`, and
print the status code of the response it returns.


In [15]:
# your code here


## Common errors

### ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [16]:
try:
    requests.get(f"{BASE}/hang-up", timeout=5)
except ConnectionError:
    print("the connection failed, so try again later")


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

The `except` named `ConnectionError`, the exception was a `ConnectionError`, and it went straight
through. There are two classes with that name: Python's built-in `ConnectionError`, which the bare
name means, and `requests.exceptions.ConnectionError`, which requests raises, and which is not a kind
of the built-in one. Name requests' class in full:


In [17]:
print("requests' ConnectionError is a built-in ConnectionError:",
      issubclass(requests.exceptions.ConnectionError, ConnectionError))

try:
    requests.get(f"{BASE}/hang-up", timeout=5)
except requests.exceptions.ConnectionError:
    print("the connection failed, so try again later")


requests' ConnectionError is a built-in ConnectionError: False
the connection failed, so try again later


### No error, and a 404 asked for four times: a retry for a failure that cannot pass


In [18]:
statuses = []
for attempt in range(4):
    response = requests.get(f"{BASE}/stations/narvik", timeout=5)
    statuses.append(response.status_code)
    if response.ok:
        break
    time.sleep(0.5)

print(statuses)


[404, 404, 404, 404]


The loop retried every failure, and a station that does not exist does not start existing half a
second later. Four requests and two seconds of waiting bought the answer the first request gave. Ask
whether a failure is worth retrying before waiting for it:


In [19]:
response = get_with_retries(f"{BASE}/stations/narvik", rng=random.Random(1))
print(response.status_code)


  attempt 1: 404
404


### ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=8765): Read timed out. (read timeout=1)


In [20]:
response = requests.get(f"{BASE}/network/report", timeout=(10, 1))      # meant: 10 seconds for the report


ReadTimeout: HTTPConnectionPool(host='127.0.0.1', port=8765): Read timed out. (read timeout=1)

In a pair, the first number is the connect timeout and the second the read timeout, so this request
allowed 10 seconds to connect and 1 second for the report, which takes 2. Connecting is usually the
quick part, so the longer number goes second:


In [21]:
response = requests.get(f"{BASE}/network/report", timeout=(3.05, 10))
print(response.status_code, response.json())


200 {'stations': 4, 'reporting': 3, 'readings': 216, 'events': 54}


### No error, and 3 seconds under a timeout of 1: a timeout taken for a deadline


In [22]:
start = time.monotonic()
response = requests.get(f"{BASE}/trickle", timeout=1)

print(response.status_code, "after", round(time.monotonic() - start), "seconds, with a timeout of 1")


200 after 3 seconds, with a timeout of 1


The read timeout limits every wait for the next piece of a response, and `/trickle` sent a piece
every half second, so no wait reached a second while the whole response took three. The **Your First
Request** notebook said a timeout is not a limit on the whole request, and this is what that looks
like. A program that needs an answer within a time keeps a deadline of its own. Between attempts,
`get_with_retries` checks one. Within a single response, `stream=True` leaves the body unread until
the program asks for it, and `iter_content` hands it over in chunks of the size asked for, here a
byte, so the program can check the clock as it reads:


In [23]:
start = time.monotonic()
with requests.get(f"{BASE}/trickle", timeout=1, stream=True) as response:
    body = b""
    for piece in response.iter_content(chunk_size=1):
        body += piece
        if time.monotonic() - start > 1.2:
            print("stopped reading after", len(body), "of", response.headers["Content-Length"], "bytes: past the deadline")
            break


stopped reading after 23 of 62 bytes: past the deadline


### RetryError: HTTPConnectionPool(host='127.0.0.1', port=8765): Max retries exceeded with url: /status/502 (Caused by ResponseError('too many 502 error responses'))


In [24]:
session.get(f"{BASE}/status/502", timeout=5)


RetryError: HTTPConnectionPool(host='127.0.0.1', port=8765): Max retries exceeded with url: /status/502 (Caused by ResponseError('too many 502 error responses'))

Every attempt got `502`. When the retries ran out, `Retry` raised `RetryError` instead of returning the
last response, so code that checks `status_code` never sees one. With `raise_on_status=False`, the
last response comes back, to be decided about like any other:


In [25]:
patient = requests.Session()
patient.mount("http://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=0.5, status_forcelist=[502, 503, 504],
                                                       raise_on_status=False)))
response = patient.get(f"{BASE}/status/502", timeout=5)
print(response.status_code, "after", len(response.raw.retries.history), "retries")


502 after 3 retries


## Recap

- A request can fail without a response: a `Timeout` when a server is too slow, a `ConnectionError`
  when a connection fails or closes. `requests.exceptions.RequestException` catches both, and
  everything else requests raises.
- `timeout=(connect, read)` limits the wait for a connection and the wait between pieces of a
  response, never the whole request, so keep a deadline for that.
- Retry timeouts, connection errors, `408`, `429`, `500`, `502`, `503` and `504`, and never a `400`,
  `401`, `403` or `404`.
- Retry only a request that is safe to send twice, such as a `GET`, and send the same request id on
  every attempt.
- Wait between attempts with backoff and jitter, or as `Retry-After` asks, and stop at a most number of
  attempts and a deadline.
- urllib3's `Retry`, mounted on a `Session`, retries for you, and raises `RetryError` when it runs out,
  unless `raise_on_status=False`.


## What is next

The **Sending Data** notebook. Every request here only read, so sending one twice did no harm. That
notebook sends data that changes things, with `POST`, `PUT` and `DELETE`, and shows what idempotent
means in practice, when a request that creates something may be sent twice.


---

&#8592; **Previous:** [Rate Limits](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/11-rate-limits.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Sending Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/13-sending-data.ipynb) &#8594;
